In [ ]:
import os
import shutil

# 0. Bulletproof Setup: Check for repo and enforce working directory
repo_path = "/kaggle/working/rl_sf"
if not os.path.exists(repo_path):
    print("--> Repository missing. Cloning now...")
    !git clone -b optimization https://github.com/flaviogeuforbio/rl-with-sf-for-mujoco {repo_path}

# Change directory explicitly to where the script lives
%cd {repo_path}

In [ ]:
import torch
print("torch:", torch.version)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
!python -m pip install mujoco

In [ ]:
%ls

!ls -l /kaggle/input/

In [ ]:
!ls -R /kaggle/input/datasets

In [ ]:
import os
import shutil

# --- PHASE 2 : Checkpointing ---
seed = 1
folder_name = f"seed_{seed}_scratch" # Dynamic folder name

# For a quick safety check
# STEPS_PER_PHASE = "4000"
# RUN_STEPS_LIMIT = "1000"   
# SAVE_FREQ = "200"

STEPS_PER_PHASE = "4000000"
RUN_STEPS_LIMIT = "1000000"   
SAVE_FREQ = "200000"

GAMMA_VAL = "0.99"
LAMBDA_Q = "1.0" 
LAMBDA_VEC = "1.0" 
RESUME_DIR = "/kaggle/input/datasets/domenicoscarlatti/checkpointwalkeronlyseed1" # change the name of the last folder according to the name you give to the dataset...

RUN_NAME_SEQ = f"Walker_from_scratch_gamma_{GAMMA_VAL.replace('.', '_')}_lq_{LAMBDA_Q.replace('.', '_')}_lvec_{LAMBDA_VEC.replace('.', '_')}_stepsxphase_{STEPS_PER_PHASE}"

kaggle_output_folder = f"/kaggle/working/{folder_name}"
os.makedirs(kaggle_output_folder, exist_ok=True)

local_path_seq = f"/kaggle/working/rl_sf/artifacts/walker/{RUN_NAME_SEQ}"


In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    if 'checkpoint_sf.pt' in files:
        print(f"FOUND IT! Set your RESUME_DIR to:\n{root}")

In [ ]:
print(f"\n--- EXECUTING SEED {seed} ---")

print("-> Running Sequential Training...")
!python -u train_cheetah_walker.py --steps_per_phase {STEPS_PER_PHASE} --save_freq {SAVE_FREQ} --baseline --run_name {RUN_NAME_SEQ} --gamma {GAMMA_VAL} --lambda_q {LAMBDA_Q} --lambda_vec {LAMBDA_VEC} --seed {seed} --resume_dir {RESUME_DIR} --run_steps_limit {RUN_STEPS_LIMIT} --walker_only

if os.path.exists(local_path_seq):
    final_dest_seq = os.path.join(kaggle_output_folder, RUN_NAME_SEQ)
    shutil.copytree(local_path_seq, final_dest_seq, dirs_exist_ok=True)
    print(f"--> Scratch learning data (walker only) saved in: {final_dest_seq}")

print("\n--> Zipping results for download...")
%cd /kaggle/working/
!zip -r {folder_name}.zip {folder_name}/
%cd /kaggle/working/rl_sf